# OpenMythos on a Colab T4

Trains the `mini_60m` reference model (~62.9M params) on TinyStories.
Run cells top to bottom. Uses a T4 GPU (Runtime > Change runtime type > T4).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Get the code: clone your published repo, or upload openmythos.zip
REPO_URL = ""  # e.g. https://github.com/jayantt1dubeyy/openmythos
import os
if REPO_URL:
    !git clone $REPO_URL
    %cd openmythos
else:
    print("Upload openmythos.zip via the Files panel, then run the next cell")

In [ ]:
!unzip -q -o /content/openmythos.zip -d /content 2>/dev/null; %cd /content/openmythos 2>/dev/null || %cd /content
!pip install -q -r requirements.txt
!pip install -q datasets

In [ ]:
# Sanity checks before the long run
!python -m openmythos.benchmark --preset tiny_debug --mode train --steps 5
!python -m openmythos.benchmark --preset mini_60m --mode vram

In [ ]:
# Data: TinyStories
from datasets import load_dataset
import os
os.makedirs("data", exist_ok=True)
for split, name in [("train", "train.txt"), ("validation", "val.txt")]:
    ds = load_dataset("roneneldan/TinyStories", split=split)
    with open(f"data/{name}", "w") as f:
        for ex in ds:
            f.write(ex["text"].strip() + "\n")
    print(name, "written")

In [ ]:
# Train in the background so the cell returns immediately
!nohup python -m openmythos.train --preset mini_60m --train-data data/train.txt --val-data data/val.txt > train.log 2>&1 &
print("training started — watch with: !tail -5 train.log")

In [ ]:
!tail -5 train.log

In [ ]:
# Resume after preemption:
# !nohup python -m openmythos.train --preset mini_60m --train-data data/train.txt --val-data data/val.txt --resume checkpoints/latest.pt > train.log 2>&1 &

In [ ]:
# Evaluate
!python -m openmythos.evaluate --ckpt checkpoints/final.pt --text data/val.txt

In [ ]:
# Generate
import torch
from openmythos.config import ModelConfig
from openmythos.model import OpenMythosModel
from openmythos.tokenizer import ByteTokenizer
from openmythos.inference import generate
ckpt = torch.load("checkpoints/final.pt", map_location="cuda", weights_only=False)
model = OpenMythosModel(ModelConfig.from_json(ckpt["model_cfg"])).cuda().eval()
model.load_state_dict(ckpt["model"])
print(generate(model, ByteTokenizer(), "Once upon a time", max_new_tokens=100, device="cuda"))